In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from collections import Counter
from sklearn.datasets import load_iris

# 1. Gini Impurity (Same as before)
def gini_impurity(y):
    m = len(y)
    if m == 0: return 0.0
    probabilities = np.bincount(y) / m
    return 1.0 - np.sum(probabilities ** 2)

# 2. Enhanced Node Class
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, 
                 value=None, gini=None, samples=None, gini_decrease=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        
        # --- NEW: Tracking Metadata ---
        self.gini = gini                  # Gini of the node BEFORE splitting
        self.samples = samples            # Number of samples in this node
        self.gini_decrease = gini_decrease # How much impurity this split removed

# 3. Enhanced CART Class
class CustomCART_Verbose:
    def __init__(self, min_samples_split=2, max_depth=10, verbose=False):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.verbose = verbose
        self.root = None

    def fit(self, X, y):
        self.root = self._grow_tree(X, y, depth=0)

    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        current_gini = gini_impurity(y)

        # Stopping Criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            if self.verbose:
                print(f"{'  ' * depth}Leaf Node Reached | Samples: {n_samples} | Gini: {current_gini:.3f} | Pred: Class {leaf_value}")
            return Node(value=leaf_value, gini=current_gini, samples=n_samples)

        # Greedy Search for Best Split
        best_feat, best_thresh, best_split_gini = self._best_split(X, y, n_features, depth)
        
        if best_feat is None:
             return Node(value=self._most_common_label(y), gini=current_gini, samples=n_samples)

        # Calculate Information Gain (Gini Decrease)
        gini_decrease = current_gini - best_split_gini

        # Split and Recurse
        lhs_mask = X[:, best_feat] < best_thresh
        rhs_mask = ~lhs_mask
        
        left_child = self._grow_tree(X[lhs_mask, :], y[lhs_mask], depth + 1)
        right_child = self._grow_tree(X[rhs_mask, :], y[rhs_mask], depth + 1)
        
        return Node(feature_index=best_feat, threshold=best_thresh, left=left_child, right=right_child, 
                    gini=current_gini, samples=n_samples, gini_decrease=gini_decrease)

    def _best_split(self, X, y, n_features, depth):
        best_gini = float('inf')
        best_feat, best_thresh = None, None

        for feat_idx in range(n_features):
            unique_vals = np.sort(np.unique(X[:, feat_idx]))
            split_locations = (unique_vals[:-1] + unique_vals[1:]) / 2

            for threshold in split_locations:
                lhs_mask = X[:, feat_idx] < threshold
                y_lhs, y_rhs = y[lhs_mask], y[~lhs_mask]
                
                n_lhs, n_rhs = len(y_lhs), len(y_rhs)
                if n_lhs == 0 or n_rhs == 0: continue
                
                g_split = (n_lhs/len(y)) * gini_impurity(y_lhs) + (n_rhs/len(y)) * gini_impurity(y_rhs)

                if g_split < best_gini:
                    best_gini = g_split
                    best_feat = feat_idx
                    best_thresh = threshold

        if self.verbose and best_feat is not None:
             print(f"{'  ' * depth}Depth {depth} Split | Feat {best_feat} <= {best_thresh:.2f} | Weighted Gini: {best_gini:.3f}")

        return best_feat, best_thresh, best_gini

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.value is not None: return node.value
        if x[node.feature_index] < node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

# Load data
iris = load_iris()
X = iris.data[:, 2:4] # Petal Length and Width
y = iris.target

# Train with Verbose enabled to watch the greedy algorithm work
print("--- Watching the Greedy Algorithm Layer-by-Layer ---")
tree_tracker = CustomCART_Verbose(max_depth=3, verbose=True)
tree_tracker.fit(X, y)

--- Watching the Greedy Algorithm Layer-by-Layer ---
Depth 0 Split | Feat 0 <= 2.45 | Weighted Gini: 0.333
  Leaf Node Reached | Samples: 50 | Gini: 0.000 | Pred: Class 0
  Depth 1 Split | Feat 1 <= 1.75 | Weighted Gini: 0.110
    Depth 2 Split | Feat 0 <= 4.95 | Weighted Gini: 0.086
      Leaf Node Reached | Samples: 48 | Gini: 0.041 | Pred: Class 1
      Leaf Node Reached | Samples: 6 | Gini: 0.444 | Pred: Class 2
    Depth 2 Split | Feat 0 <= 4.85 | Weighted Gini: 0.029
      Leaf Node Reached | Samples: 3 | Gini: 0.444 | Pred: Class 2
      Leaf Node Reached | Samples: 43 | Gini: 0.000 | Pred: Class 2


In [2]:
def print_tree_structure(node, feature_names, depth=0):
    indent = "    " * depth
    if node.value is not None:
        print(f"{indent}└── Leaf: Predict Class {node.value} (Gini: {node.gini:.3f}, Samples: {node.samples})")
    else:
        print(f"{indent}├── IF {feature_names[node.feature_index]} <= {node.threshold:.2f}")
        print(f"{indent}│   (Node Gini: {node.gini:.3f}, Samples: {node.samples} | Gini Decrease: -{node.gini_decrease:.3f})")
        print_tree_structure(node.left, feature_names, depth + 1)
        
        print(f"{indent}├── ELSE")
        print_tree_structure(node.right, feature_names, depth + 1)

print("\n--- Final Tree Structure & Gini Tracking ---")
print_tree_structure(tree_tracker.root, feature_names=["Petal Length", "Petal Width"])


--- Final Tree Structure & Gini Tracking ---
├── IF Petal Length <= 2.45
│   (Node Gini: 0.667, Samples: 150 | Gini Decrease: -0.333)
    └── Leaf: Predict Class 0 (Gini: 0.000, Samples: 50)
├── ELSE
    ├── IF Petal Width <= 1.75
    │   (Node Gini: 0.500, Samples: 100 | Gini Decrease: -0.390)
        ├── IF Petal Length <= 4.95
        │   (Node Gini: 0.168, Samples: 54 | Gini Decrease: -0.082)
            └── Leaf: Predict Class 1 (Gini: 0.041, Samples: 48)
        ├── ELSE
            └── Leaf: Predict Class 2 (Gini: 0.444, Samples: 6)
    ├── ELSE
        ├── IF Petal Length <= 4.85
        │   (Node Gini: 0.043, Samples: 46 | Gini Decrease: -0.014)
            └── Leaf: Predict Class 2 (Gini: 0.444, Samples: 3)
        ├── ELSE
            └── Leaf: Predict Class 2 (Gini: 0.000, Samples: 43)
